[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C47_RecSys_Ranking_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与数据加载基座

本课全程 **纯 numpy、CPU 可跑**，从零实现推荐/排序算法，用 **真实 MovieLens-like 数据** 验证。

这个 notebook 做四件事：① 确认环境；② 给出**全课通用的数据加载器**（真实 MovieLens 优先，联网失败回退到可复现合成数据）；③ 用一个最小例子体会**显式 vs 隐式反馈**与**稀疏矩阵**；④ 立下全课纪律——**对拍 + 看指标**。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画收敛/指标曲线）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 全课通用：MovieLens 数据加载器（真实优先，失败回退合成）

这是**整门课都会复用**的数据基座。它先尝试加载真实 MovieLens-100k（本地文件或下载），
**联网/文件失败则回退**到一个可复现的合成评分数据——合成数据带真实的结构（用户偏置、物品偏置、低秩隐因子 + 噪声），
所以后面所有算法、对拍、指标在两种数据上都成立。两条路返回**完全一致的接口**：`(ratings, n_users, n_items)`，
其中 `ratings` 是 `(user, item, rating)` 的整数/浮点数组。

In [ ]:
def load_movielens_or_synth(n_users=200, n_items=300, rank=8, seed=0, verbose=True):
    '''返回 (ratings, n_users, n_items)。ratings: (N,3) float 数组 [user, item, rating(1~5)].
       优先真实 MovieLens-100k；失败则生成可复现的合成数据（低秩 + 偏置 + 噪声）。'''
    # ---- 尝试真实 MovieLens-100k ----
    import os
    for path in ['ml-100k/u.data', 'u.data', os.path.expanduser('~/ml-100k/u.data')]:
        if os.path.exists(path):
            data = np.loadtxt(path, dtype=np.int64)[:, :3].astype(float)
            data[:, 0] -= 1; data[:, 1] -= 1     # 1-indexed -> 0-indexed
            nu = int(data[:, 0].max()) + 1; ni = int(data[:, 1].max()) + 1
            if verbose: print(f'已加载真实 MovieLens-100k：{len(data)} 条评分, {nu} 用户, {ni} 电影')
            return data, nu, ni
    try:
        import urllib.request, zipfile, io
        url = 'https://files.grouplens.org/datasets/movielens/ml-100k/u.data'
        with urllib.request.urlopen(url, timeout=5) as r:
            raw = r.read().decode()
        rows = [list(map(int, ln.split('\t')[:3])) for ln in raw.strip().split('\n')]
        data = np.array(rows, dtype=float); data[:, 0] -= 1; data[:, 1] -= 1
        nu = int(data[:, 0].max()) + 1; ni = int(data[:, 1].max()) + 1
        if verbose: print(f'已下载真实 MovieLens-100k：{len(data)} 条评分')
        return data, nu, ni
    except Exception as e:
        if verbose: print(f'联网/文件失败（{type(e).__name__}），回退到可复现合成数据。')
    # ---- 回退：合成低秩评分（带真实结构）----
    rng = np.random.default_rng(seed)
    P = rng.standard_normal((n_users, rank)) * 0.5     # 用户隐因子
    Q = rng.standard_normal((n_items, rank)) * 0.5     # 物品隐因子
    bu = rng.standard_normal(n_users) * 0.3            # 用户偏置（有人爱打高分）
    bi = rng.standard_normal(n_items) * 0.5            # 物品偏置（有的电影普遍受欢迎）
    mu = 3.5                                            # 全局均值
    # 每个用户随机评 ~15% 的电影（模拟稀疏 + 选择偏置）
    rows = []
    for u in range(n_users):
        k = rng.integers(20, 60)
        items = rng.choice(n_items, size=min(k, n_items), replace=False)
        for i in items:
            r = mu + bu[u] + bi[i] + P[u] @ Q[i] + rng.standard_normal() * 0.3
            r = float(np.clip(np.round(r * 2) / 2, 1.0, 5.0))  # 截断到 1~5（半星）
            rows.append([u, i, r])
    data = np.array(rows, dtype=float)
    if verbose: print(f'已生成合成评分：{len(data)} 条, {n_users} 用户, {n_items} 电影（seed={seed}, 可复现）')
    return data, n_users, n_items

ratings, n_users, n_items = load_movielens_or_synth(seed=0)
print('ratings.shape =', ratings.shape, '| 列含义 = [user, item, rating]')
print('评分范围:', ratings[:, 2].min(), '~', ratings[:, 2].max(), '| 均值:', round(ratings[:, 2].mean(), 3))
assert ratings.shape[1] == 3 and len(ratings) > 1000
assert ratings[:, 2].min() >= 1.0 and ratings[:, 2].max() <= 5.0
print('✅ 数据基座就绪（真实或合成，接口一致）')

## 3 · 交互矩阵与稀疏度

推荐的核心数据结构是 **用户-物品交互矩阵** $R$（行=用户、列=物品）。它的灵魂特征是**极度稀疏**——
多数 (用户,物品) 对从未交互。下面把评分列表转成矩阵，量化稀疏度。

In [ ]:
def to_dense_matrix(ratings, n_users, n_items):
    '''把 (user,item,rating) 列表转成稠密矩阵 R，未观测处为 0（仅用于教学；真实工业用稀疏存储）。'''
    R = np.zeros((n_users, n_items))
    R[ratings[:, 0].astype(int), ratings[:, 1].astype(int)] = ratings[:, 2]
    return R

R = to_dense_matrix(ratings, n_users, n_items)
observed = (R > 0).sum()
total = n_users * n_items
sparsity = 1 - observed / total
print(f'矩阵形状: {R.shape}, 已观测: {observed}, 总格子: {total}')
print(f'稀疏度 = {sparsity:.4f}  (即 {sparsity*100:.2f}% 的格子是空的)')
assert sparsity > 0.5, '推荐矩阵必然高度稀疏'
print('✅ 稀疏是推荐区别于普通监督学习的根本特征：我们要从极少的观测里学会填空/排序')

## 4 · 显式 → 隐式：同一份数据的两种用法

**显式反馈**：直接用评分（回归目标，预测 r_ui）。
**隐式反馈**：把「交互过」当正例（二值 1），「未交互」当（候选）负例——只有正例的单类问题。

下面把同一份评分既看成显式（保留分值）又看成隐式（二值化），体会两者的区别。

In [ ]:
# 显式：保留评分
explicit_vals = ratings[:, 2]
print('显式反馈：评分分布', {v: int((explicit_vals == v).sum()) for v in sorted(set(explicit_vals))[:6]}, '...')

# 隐式：二值化（>=4 星算正向交互，模拟「喜欢」）；其余视为未观测/弱信号
implicit_pos = ratings[ratings[:, 2] >= 4.0][:, :2].astype(int)   # (user,item) 正例对
print(f'隐式反馈：{len(implicit_pos)} 个正例对（>=4星）；其余 {total - len(implicit_pos)} 对是「未观测」（既非确定正也非确定负）')

# 关键：未观测 != 负反馈
n_unobserved = total - observed
print(f'\n注意：{n_unobserved} 个完全未交互的对里，有多少是「不喜欢」、多少是「没看到」？我们无从分辨。')
assert len(implicit_pos) < observed, '正例(>=4星) 应少于全部观测'
print('✅ 这就是隐式反馈的单类难题：只有正例，负例要靠采样构造（模块 03/04 详解）')

## 5 · 按时间/留一切分：避免时间泄漏

推荐评估**必须尊重时间因果**：不能用未来预测过去。最常用的是 **leave-last-out**——
每个用户留出**最后一次**交互做测试，其余做训练。这里没有时间戳就用「每个用户随机留一个」近似（真实数据按时间留）。

下面实现 leave-one-out 切分，它会被后面 Top-N 评估反复用到。

In [ ]:
def leave_one_out_split(ratings, seed=0):
    '''每个用户留出一条交互做测试，其余训练。返回 (train, test) 两个 (N,3) 数组。'''
    rng = np.random.default_rng(seed)
    train, test = [], []
    for u in np.unique(ratings[:, 0]):
        idx = np.where(ratings[:, 0] == u)[0]
        if len(idx) < 2:
            train.extend(ratings[idx]); continue
        held = rng.choice(idx)                 # 留一（真实数据应取时间最晚的一条）
        test.append(ratings[held])
        train.extend(ratings[i] for i in idx if i != held)
    return np.array(train), np.array(test)

train, test = leave_one_out_split(ratings, seed=0)
print(f'训练集 {len(train)} 条, 测试集 {len(test)} 条')
# 检验：测试集里每个用户恰好一条；训练+测试 = 全部（对有>=2条的用户）
test_users = test[:, 0]
assert len(test_users) == len(set(test_users)), '每个用户测试集最多一条'
assert len(train) + len(test) == len(ratings)
print('✅ 留一切分正确：训练/测试不重叠，每用户留一条评估。绝不用未来预测过去！')

## 6 · 全课纪律：对拍 + 看指标

本课每个算法要么**对拍**一个朴素参考实现（数值一致），要么**验证损失/指标单调改善**（收敛）。
先把两个最常用的工具立起来：一个对拍函数、一个最简 Recall@K。后面模块都用它们当裁判。

In [ ]:
def check_allclose(name, got, ref, atol=1e-8):
    '''对拍：被测实现 vs 朴素参考。返回是否一致并打印。'''
    ok = np.allclose(got, ref, atol=atol)
    max_err = float(np.max(np.abs(np.asarray(got) - np.asarray(ref)))) if np.size(got) else 0.0
    print(f'[{name:<28}] allclose={ok}  max|err|={max_err:.2e}')
    assert ok, f'{name} 与参考不一致！'
    return ok

def recall_at_k(ranked_items, relevant_set, k):
    '''Top-K 里命中的相关物品数 / 全部相关物品数。'''
    if not relevant_set:
        return 0.0
    topk = ranked_items[:k]
    hits = sum(1 for it in topk if it in relevant_set)
    return hits / len(relevant_set)

# 演示：一个完美排序 vs 一个随机排序
relevant = {2, 5, 9}
perfect = [2, 5, 9, 0, 1, 3, 4, 6, 7, 8]      # 相关的排最前
bad     = [0, 1, 3, 4, 6, 2, 7, 8, 5, 9]      # 相关的排很后
print('完美排序 Recall@3 =', recall_at_k(perfect, relevant, 3))
print('糟糕排序 Recall@3 =', recall_at_k(bad, relevant, 3))
assert recall_at_k(perfect, relevant, 3) == 1.0
assert recall_at_k(bad, relevant, 3) < 0.5
check_allclose('对拍工具自检', np.array([1.0, 2.0]), np.array([1.0, 2.0]))
print('\n✅ 裁判就位：对拍保证「算对」，Recall@K 等指标衡量「排得好」。')

## 7 · ✏️ 练习：实现 Precision@K 与 Hit@K

你已经看过 `recall_at_k`。照葫芦实现另外两个 Top-N 指标：
- `precision_at_k`：Top-K 里命中的相关物品数 ÷ **K**
- `hit_at_k`：Top-K 里**是否至少**命中一个相关物品（返回 0.0 或 1.0）

In [ ]:
def precision_at_k(ranked_items, relevant_set, k):
    # TODO: Top-K 命中数 / k
    raise NotImplementedError

def hit_at_k(ranked_items, relevant_set, k):
    # TODO: Top-K 是否至少命中一个 -> 1.0 / 0.0
    raise NotImplementedError

In [ ]:
# —— 练习 自测 ——
relevant = {2, 5, 9}
perfect = [2, 5, 9, 0, 1, 3, 4, 6, 7, 8]
bad     = [0, 1, 3, 4, 6, 2, 7, 8, 5, 9]
assert abs(precision_at_k(perfect, relevant, 3) - 1.0) < 1e-9
assert abs(precision_at_k(perfect, relevant, 5) - 3/5) < 1e-9   # 5 个里命中 3 个
assert hit_at_k(perfect, relevant, 1) == 1.0
assert hit_at_k(bad, relevant, 3) == 0.0                         # 前3个没命中
assert hit_at_k(bad, relevant, 6) == 1.0                         # 前6个命中了 item 2
print('✅ 练习通过：你实现了 Precision@K 与 Hit@K')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
def precision_at_k(ranked_items, relevant_set, k):
    if k == 0:
        return 0.0
    topk = ranked_items[:k]
    hits = sum(1 for it in topk if it in relevant_set)
    return hits / k

def hit_at_k(ranked_items, relevant_set, k):
    topk = ranked_items[:k]
    return 1.0 if any(it in relevant_set for it in topk) else 0.0
print('参考答案已载入')

## 8 · 🧪 真实数据胶囊：MovieLens 的统计画像

用刚加载的真实/合成 MovieLens 数据，算几个推荐工程师天天要看的统计量：**每用户平均评分数、每物品平均被评数、长尾分布**。
这些数字决定了召回/排序的难度（越稀疏越难、长尾越重冷启动越严重）。

**TODO**：补全 `data_profile`，统计每用户交互数与每物品交互数的均值、中位数、最大值。

In [ ]:
def data_profile(ratings, n_users, n_items):
    '''返回 dict：用户/物品的交互数统计。'''
    user_counts = np.bincount(ratings[:, 0].astype(int), minlength=n_users)
    item_counts = np.bincount(ratings[:, 1].astype(int), minlength=n_items)
    # TODO: 填入 user_counts / item_counts 的 mean, median, max
    return {
        'user_mean': ...,   'user_median': ...,   'user_max': ...,
        'item_mean': ...,   'item_median': ...,   'item_max': ...,
    }

In [ ]:
# 自测（胶囊）
prof = data_profile(ratings, n_users, n_items)
print('每用户评分数: mean={user_mean:.1f} median={user_median:.0f} max={user_max}'.format(**prof))
print('每物品被评数: mean={item_mean:.1f} median={item_median:.0f} max={item_max}'.format(**prof))
assert prof['user_mean'] > 0 and prof['item_max'] >= prof['item_median']
print('✅ 胶囊通过：物品被评数的 max >> median，即典型的长尾——少数热门占据大量交互，长尾物品冷启动难。')

In [ ]:
# 📖 胶囊参考答案
def data_profile(ratings, n_users, n_items):
    uc = np.bincount(ratings[:, 0].astype(int), minlength=n_users)
    ic = np.bincount(ratings[:, 1].astype(int), minlength=n_items)
    return {
        'user_mean': float(uc.mean()), 'user_median': float(np.median(uc)), 'user_max': int(uc.max()),
        'item_mean': float(ic.mean()), 'item_median': float(np.median(ic)), 'item_max': int(ic.max()),
    }

### 小结
- 推荐/搜索/排序 = 工业界最大的 ML 就业面；核心是 **召回→粗排→精排→重排** 的级联漏斗，每层目标/指标不同。
- 数据是 **稀疏的用户-物品交互矩阵**；**显式反馈**（评分，有正负）vs **隐式反馈**（行为，只有正例 + 噪声）。
- 评估用 **Top-N 排序指标**（Recall/Precision/Hit/nDCG/MAP），且**必须按时间切分**避免泄漏；离线指标只是代理，线上 A/B 才是裁判。
- 召回层 = 向量检索，与 **C11（RAG）同源**（双塔 + ANN）。

下一站：**模块 01 · 协同过滤** —— 不用任何物品内容，只靠「谁和谁交互过」就能推荐。